#### Define notebook Parameters

Pass through th pl_worker

In [1]:
# Framework-compatible parameters with manual fallbacks.
# When pl_worker calls this notebook, it should pass these values.
import json
import uuid

# Basic notebook runtime values
task_id = 50
task_name = "Load RMDD to silver"
task_executions_id = ""
parent_run_id = ""
lineage_id = str(uuid.uuid4())
limit_rows_for_debugging = False
#prev_wm = "1900-01-01 00:00:00"
prev_wm = "2026-06-01 00:00:00"
curr_wm = "2026-06-30 00:00:00"
server_name = "hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com"

# Framework-style placeholder; real connection settings should come from pipeline/metadata
source_connection_settings = "{}"

# Source config: one database, many tables to load
source_settings = json.dumps({
    "source_name": "CAN_RMDD",
    "database_name": "CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e",
    "table_config": [
        {
            "schema_name": "dbo",
            "table_name": "Address",
            "target_table": "silver_rmdd_address",
            "primary_keys": ["address_id"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "table_name": "Client",
            "target_table": "silver_rmdd_client",
            "primary_keys": ["client_number", "member_firm_code"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "table_name": "Person",
            "target_table": "silver_rmdd_person",
            "primary_keys": ["person_number", "member_firm_code"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        },
        {
            "schema_name": "dbo",
            "table_name": "PersonRelationship",
            "target_table": "silver_rmdd_person_relationship",
            "primary_keys": ["person_number_1", "member_firm_code_1", "person_number_2", "member_firm_code_2"],
            "is_incremental": 1,
            "incremental_column": "LastUpdatedDateGMT"
        }
    ]
})

# Target config: common silver destination settings
target_settings = json.dumps({
    "lakehouse_name": "lh_canada_global_mds",
    "schema_name": "silver_rmdd",
    "load_strategy": "merge",
    "location_root": "Files/silver/rmdd"
})

StatementMeta(, 792a33ca-a99e-44c3-a8e1-085564c9fd6a, 3, Finished, Available, Finished, False)

#### Define functions and logging

Used for import functions

In [2]:
%run nb_transactional_shared_functions

StatementMeta(, 792a33ca-a99e-44c3-a8e1-085564c9fd6a, 8, Finished, Available, Finished, True)

In [3]:
# Standard framework logging setup.
setup_logging()
logger = logging.getLogger("LoadTransactionalToBase")
logger.setLevel(logging.INFO)

StatementMeta(, 792a33ca-a99e-44c3-a8e1-085564c9fd6a, 9, Finished, Available, Finished, False)

#### Define variables or parameters

Can manually run or be through worker

In [4]:
# Accept either JSON strings (framework) or already-parsed dicts (manual/debug).
source_settings = json.loads(source_settings) if isinstance(source_settings, str) else source_settings
target_settings = json.loads(target_settings) if isinstance(target_settings, str) else target_settings
source_connection_settings = json.loads(source_connection_settings) if isinstance(source_connection_settings, str) else source_connection_settings

# Source configuration
source_name = source_settings.get("source_name")
source_database = source_settings.get("database_name")
table_config = source_settings.get("table_config", [])

# Target configuration
target_lakehouse_name = target_settings.get("lakehouse_name")
target_schema = target_settings.get("schema_name")
target_load_strategy = target_settings.get("load_strategy", "merge")
location_root = target_settings.get("location_root")

# SQL connection configuration
server_name = (
    source_connection_settings.get("server_name")
    or source_connection_settings.get("jdbc_hostname")
    or source_connection_settings.get("server")
)

if not server_name: server_name = "hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com"
print(source_name)
print(source_database)
print(target_schema)
display(table_config)

StatementMeta(, 792a33ca-a99e-44c3-a8e1-085564c9fd6a, 10, Finished, Available, Finished, False)

CAN_RMDD
CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e
silver_rmdd
2026-06-30 20:55:49,177 UTC - INFO - Calling getAccessToken from PyTridentTokenLibrary (trident_token_library_wrapper)


SynapseWidget(Synapse.DataFrame, 8215ff25-d2cf-431f-ac5a-50d5280f0f9b)

#### Build JDBC connection

Connects to source SQL database

In [5]:
# Build one JDBC connection for the source SQL database
jdbc_url = (
    "jdbc:sqlserver://"
    f"{server_name}:1433;"
    f"database={source_database};"
    "encrypt=true;"
)

connection_properties = {
    "accessToken": mssparkutils.credentials.getToken("https://database.windows.net/"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print(jdbc_url)
print(connection_properties)

StatementMeta(, 792a33ca-a99e-44c3-a8e1-085564c9fd6a, 11, Finished, Available, Finished, False)

jdbc:sqlserver://hbvkj6b5fvzenlsxgtupezx6wq-4qzy5g26bakuhffkkiyl64azc4.database.fabric.microsoft.com:1433;database=CAN_RMDD-236c5edd-b846-437b-b99b-39e8e56a0a5e;encrypt=true;
{'accessToken': '[REDACTED]', 'driver': 'com.microsoft.sqlserver.jdbc.SQLServerDriver'}


#### Ingest source table

Read and dedup clean source

In [6]:
# Read each source table and drop exact duplicates only
for cfg in table_config:
    # Get variables from YAML
    source_schema = cfg.get("schema_name")
    source_table = cfg.get("table_name")
    column_list = cfg.get("select_columns", ["*"])
    is_incremental = cfg.get("is_incremental", 0)
    watermark_col = cfg.get("incremental_column")

    # Set source table to query
    full_source_name = f"{source_schema}.{source_table}"
    column_list_sql = ", ".join(column_list)
    source_view = f"src_{source_table.lower()}"

    # Set watermark based on YAML watermark_col
    watermark_filter = "1 = 1"
    if is_incremental == 1:
        watermark_filter = f"{watermark_col} > '{prev_wm}' AND {watermark_col} <= '{curr_wm}'"

    # Query to source with watermark applied
    source_query = f"""
    (
        SELECT {column_list_sql}
        FROM {full_source_name}
        WHERE {watermark_filter}
    ) AS src
    """

    # Read table using JDBC connection
    df_source = spark.read.jdbc(
        url=jdbc_url,
        table=source_query,
        properties=connection_properties
    )

    # Remove duplicates and create temp view for later consumption
    df_source = remove_exact_duplicates(df_source)
    df_source.createOrReplaceTempView(source_view)

    print(f"Source preview for {full_source_name}")
    display(df_source.limit(3))

StatementMeta(, 792a33ca-a99e-44c3-a8e1-085564c9fd6a, 12, Finished, Available, Finished, True)

Source preview for dbo.Address


SynapseWidget(Synapse.DataFrame, 303ef037-ac14-415b-8479-2d2489a00385)

Source preview for dbo.Client


SynapseWidget(Synapse.DataFrame, b7610ec4-0e36-4104-8c63-0dc1e0fbae32)

Source preview for dbo.Person


SynapseWidget(Synapse.DataFrame, 25c15abc-7a76-4bf9-ae1b-3aeb9fa7af41)

Source preview for dbo.PersonRelationship


SynapseWidget(Synapse.DataFrame, 138a9919-9b72-4c5d-870a-ddff3ce9e85c)

#### Transform source table

Applies mapping and transformation as needed

In [7]:
# Map source columns into the target shape
for cfg in table_config:
    # Get source variables from YAML
    source_schema = cfg.get("schema_name")
    source_table = cfg.get("table_name")

    # Set source table to query
    full_source_name = f"{source_schema}.{source_table}"
    source_view = f"src_{source_table.lower()}"
    target_view = f"tgt_{source_table.lower()}"

    # Transformation block
    if source_table == "Address":
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {target_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY AddressID) AS BIGINT) AS address_key,
            AddressID               AS address_id,
            AddressTypeCode         AS address_type_code,
            Address1                AS address_1,
            Address2                AS address_2,
            Address3                AS address_3,
            Address4                AS address_4,
            Address5                AS address_5,
            City                    AS city,
            CountryID               AS country_id,
            CAST(NULL AS STRING)    AS country_code_iso2,
            State                   AS state,
            PostalCode              AS zip_code,
            Address1                AS street_address_check,
            Active                  AS is_active,
            LastUpdatedDateGMT      AS last_updated_date_utc_at_source
        FROM {source_view}
        """)

    elif source_table == "Client":
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {target_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY ClientID) AS BIGINT) AS client_key,
            ClientNumber            AS client_number,
            MemberFirmCode          AS member_firm_code,
            ClientID                AS client_id,
            PMSClientID             AS pms_client_id,
            ClientName              AS client_name,
            AlternateClientName     AS alternate_client_name,
            ContactName             AS contact_name,
            OpenDateGMT             AS open_date_utc,
            CloseDateGMT            AS close_date_utc,
            ClientDUNS              AS client_duns,
            ClientGUDUNS            AS client_guduns,
            Comments                AS comments,
            PMSEntityType           AS pms_entity_type,
            ClientType              AS client_type,
            IsConfidential          AS is_confidential,
            SICCode                 AS sic_code,
            CAST(NULL AS STRING)    AS client_sector_key,
            SanctionsChecked_YN     AS is_sanctions_checked,
            Active                  AS is_active,
            CreatedDateGMT          AS created_date_utc_at_source,
            LastUpdatedDateGMT      AS last_updated_date_utc_at_source
        FROM {source_view}
        """)

    elif source_table == "Person":
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {target_view} AS
        SELECT
            CAST(row_number() OVER (ORDER BY PersonID) AS BIGINT) AS person_key,
            SystemEmployeeCode              AS person_number,
            MemberFirmCode                  AS member_firm_code,
            PersonID                        AS person_id,
            PersonStatusCode                AS person_status_code,
            Gender                          AS gender,
            Prefix                          AS prefix,
            FirstName                       AS first_name,
            FirstNameKnownAs                AS first_name_known_as,
            MiddleName                      AS middle_name,
            LastName                        AS last_name,
            Suffix                          AS suffix,
            Initials                        AS initials,
            PhotoPath                       AS photo_path,
            CommencementDateGMT             AS commencement_date_utc,
            DepartureDateGMT                AS departure_date_utc,
            LeaveStatus                     AS leave_status,
            SubTeamID                       AS team_key,
            JobTitle                        AS job_title,
            JobTitlePMS                     AS job_title_pms,
            PracticeGroup                   AS practice_group_code,
            CAST(NULL AS STRING)            AS practice_group_key,
            CAST(YearOfCall AS STRING)      AS year_of_call,
            LastName                        AS employee_name_reporting,
            CAST(DeleteFlag AS  BOOLEAN)    AS is_active,
            LastUpdatedDateGMT              AS last_updated_date_utc_at_source
        FROM {source_view}
        """)

    elif source_table == "PersonRelationship":
        source_query = f""" (
            SELECT
                CAST(ROW_NUMBER() OVER (
                    ORDER BY pr.PersonID1, pr.PersonID2, pr.RelationshipTypeID
                ) AS BIGINT)                    AS person_relationship_key,
                p1.SystemEmployeeCode           AS person_number_1,
                p1.MemberFirmCode               AS member_firm_code_1,
                p2.SystemEmployeeCode           AS person_number_2,
                p2.MemberFirmCode               AS member_firm_code_2,
                rt.RelationshipTypeName         AS relationship_description,
                pr.PrimaryFlag                  AS primary_flag,
                pr.Active                       AS is_active,
                pr.LastUpdatedDateGMT           AS last_updated_date_utc_at_source
            FROM dbo.PersonRelationship pr
            LEFT JOIN dbo.Person p1
                ON pr.PersonID1 = p1.PersonID
            LEFT JOIN dbo.Person p2
                ON pr.PersonID2 = p2.PersonID
            LEFT JOIN dbo.RelationshipType rt
                ON pr.RelationshipTypeID = rt.RelationshipTypeID
            WHERE {watermark_filter}
        ) AS src
        """

    else:
        spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {target_view} AS
        SELECT *
        FROM {source_view}
        """)

    print(f"Mapped preview for {target_view}")
    display(spark.table(target_view).limit(3))

StatementMeta(, 792a33ca-a99e-44c3-a8e1-085564c9fd6a, 13, Submitted, Running, Running, True)

Mapped preview for tgt_address


SynapseWidget(Synapse.DataFrame, 32ea1751-eaf5-450d-a2af-a5d466790224)

Mapped preview for tgt_client


SynapseWidget(Synapse.DataFrame, 419894a8-0f7c-4516-8067-8d9d1a317f56)

Mapped preview for tgt_person


SynapseWidget(Synapse.DataFrame, c893f160-1823-4018-a2dc-c3866fb2f30b)

Mapped preview for tgt_personrelationship


#### Add metadata to source table

Applies metadata and _hashed_pk

In [ ]:
# Add metadata and hashed key to the transformed tgt_* views
for cfg in table_config:
    # Get source variables from YAML
    source_schema = cfg.get("schema_name")
    source_table = cfg.get("table_name")
    target_table = cfg.get("target_table")
    primary_keys = cfg.get("primary_keys", [])

    # Set source table to query
    full_source_name = f"{source_schema}.{source_table}"
    target_view = f"tgt_{source_table.lower()}"

    # Add metadata to source table
    df_target = spark.table(target_view)
    df_target = add_metadata(
        df=df_target,
        ingest_date=str(date.today()),
        file_path=full_source_name,
        schema_name=source_name,
        table_name=target_table,
        lineage_id=lineage_id
    )

    # Get primary columns based on YAML file
    pk_expr_cols = [to_snake_case(col) for col in primary_keys]

    # Create _hashed_pk
    df_target = df_target.withColumn(
        "_hashed_pk",
        F.md5(F.concat_ws("|", *[F.col(col).cast("string") for col in pk_expr_cols]))
    )
    df_target.createOrReplaceTempView(target_view)

    print(f"Final preview for {full_source_name}")
    display(df_target.limit(3))

StatementMeta(, , -1, Waiting, , Waiting, True)

#### Check duplicates

Return duplicates before merge 

In [ ]:
# Check for duplicate hashed keys before merge
for cfg in table_config:
    source_table = cfg.get("table_name")
    target_view = f"tgt_{source_table.lower()}"

    print(f"Checking duplicates for {target_view}")
    validate_duplicates(spark.table(target_view), "_hashed_pk", 10)

StatementMeta(, , -1, Waiting, , Waiting, True)

#### Merge to target

Merge to target table based on _hashed_key

In [ ]:
# Merge each final temp view into its target silver table
for cfg in table_config:
    source_table = cfg.get("table_name")
    target_table = cfg.get("target_table")
    target_view = f"tgt_{source_table.lower()}"
    target_table_path = f"{location_root}/{target_table}"
    full_target_name = f"{target_schema}.{target_table}"

    # Write delta files to path
    df_target = spark.table(target_view)
    load_to_target(df_target, target_table_path, target_load_strategy)

    # Create table based on delta files
    # spark.sql(f"""
    # CREATE TABLE IF NOT EXISTS {full_target_name}
    # USING DELTA
    # LOCATION '{target_table_path}'
    # """)

    # Return operationMetrics > # rows affected
    delta_table = DeltaTable.forPath(spark, target_table_path)
    operation_metrics = delta_table.history(1).select("operationMetrics").collect()[0]["operationMetrics"]

    rows_inserted = int(operation_metrics.get("numTargetRowsInserted", operation_metrics.get("numOutputRows", 0)))
    rows_updated = int(operation_metrics.get("numTargetRowsUpdated", 0))
    rows_affected = rows_inserted + rows_updated

    print(full_target_name)
    print(f"rows_affected: {rows_affected}")

StatementMeta(, , -1, Waiting, , Waiting, True)